# Early Warning for Student Attrition
## An Explainable and Fair Machine Learning Approach

### 03 — Feature Engineering

This notebook turns the EDA findings into model-ready feature sets without training models yet.

Goals:
1. preserve enrollment-stage vs Semester 1 prediction windows
2. clean feature names while keeping raw data unchanged
3. define categorical, ordinal, numerical, and fairness-related variables
4. create interpretable Semester 1 features
5. identify rare categories for later pipeline treatment
6. save reproducible processed datasets for modeling


## 1. Import Libraries


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)


## 2. Environment Setup

When running this notebook in Google Colab, the project repository is cloned so the data and project folders are available in the session.

In [5]:
from pathlib import Path

repo_dir = Path("/content/student-dropout-capstone")

if not repo_dir.exists():
    !git clone https://github.com/rayvaril/student-dropout-capstone.git

%cd /content/student-dropout-capstone/notebooks

/content/student-dropout-capstone/notebooks


## 3. Load the Raw Dataset

This supports either `data/data.csv` or the newer `data/raw/data.csv` structure.


In [7]:
data_path = Path("../data/data.csv")

if not data_path.exists():
    raise FileNotFoundError(
        "data.csv was not found in the data folder."
    )

df_raw = pd.read_csv(
    data_path,
    sep=None,
    engine="python"
)

print(f"Loaded: {data_path}")
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

Loaded: ../data/data.csv
Shape: 4,424 rows × 37 columns


## 4. Clean Column Names


In [8]:
def clean_feature_name(name):
    return (
        str(name)
        .replace("\ufeff", "")
        .replace("\t", " ")
        .strip()
    )

df = df_raw.copy()
df.columns = [clean_feature_name(col) for col in df.columns]

df.columns.tolist()


['Marital status',
 'Application mode',
 'Application order',
 'Course',
 'Daytime/evening attendance',
 'Previous qualification',
 'Previous qualification (grade)',
 'Nacionality',
 "Mother's qualification",
 "Father's qualification",
 "Mother's occupation",
 "Father's occupation",
 'Admission grade',
 'Displaced',
 'Educational special needs',
 'Debtor',
 'Tuition fees up to date',
 'Gender',
 'Scholarship holder',
 'Age at enrollment',
 'International',
 'Curricular units 1st sem (credited)',
 'Curricular units 1st sem (enrolled)',
 'Curricular units 1st sem (evaluations)',
 'Curricular units 1st sem (approved)',
 'Curricular units 1st sem (grade)',
 'Curricular units 1st sem (without evaluations)',
 'Curricular units 2nd sem (credited)',
 'Curricular units 2nd sem (enrolled)',
 'Curricular units 2nd sem (evaluations)',
 'Curricular units 2nd sem (approved)',
 'Curricular units 2nd sem (grade)',
 'Curricular units 2nd sem (without evaluations)',
 'Unemployment rate',
 'Inflation rat

## 5. Create the Binary Target

- `1` = Dropout
- `0` = Non-Dropout (`Graduate` or `Enrolled`)

The original target is preserved.


In [9]:
df["Dropout_binary"] = (df["Target"] == "Dropout").astype(int)

df[["Target", "Dropout_binary"]].head()


,Target,Dropout_binary
0,Dropout,1
1,Graduate,0
2,Dropout,1
3,Graduate,0
4,Graduate,0


## 6. Define Feature Types


In [10]:
categorical_features = [
    "Marital status",
    "Application mode",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International"
]

ordinal_features = ["Application order"]

external_context_features = [
    "Unemployment rate",
    "Inflation rate",
    "GDP"
]

semester1_raw_features = [col for col in df.columns if "1st sem" in col.lower()]
semester2_features = [col for col in df.columns if "2nd sem" in col.lower()]

print(f"Categorical features: {len(categorical_features)}")
print(f"Ordinal features: {len(ordinal_features)}")
print(f"Semester 1 raw features: {len(semester1_raw_features)}")
print(f"Semester 2 features: {len(semester2_features)}")


Categorical features: 17
Ordinal features: 1
Semester 1 raw features: 6
Semester 2 features: 6


### **Decision**

The feature groups follow the classifications established during dataset assessment.

Coded variables are treated as categorical rather than numerical, while `Application order` remains ordinal. Semester 1 and Semester 2 academic variables are kept separate so they can be controlled according to the intended prediction stage.

## 7. Define Enrollment-Stage Feature Set

`Debtor` and `Tuition fees up to date` remain timing-sensitive, so they are kept outside the conservative enrollment-stage feature set until their timing can be justified.


In [11]:
timing_pending_features = [
    "Debtor",
    "Tuition fees up to date"
]

excluded_from_enrollment = set(
    semester1_raw_features
    + semester2_features
    + ["Target", "Dropout_binary"]
)

enrollment_candidates = [
    col for col in df.columns
    if col not in excluded_from_enrollment
]

enrollment_core_features = [
    col for col in enrollment_candidates
    if col not in timing_pending_features
]

print(f"Enrollment candidates including timing-pending features: {len(enrollment_candidates)}")
print(f"Conservative enrollment core features: {len(enrollment_core_features)}")
print("Timing-pending:", timing_pending_features)


Enrollment candidates including timing-pending features: 24
Conservative enrollment core features: 22
Timing-pending: ['Debtor', 'Tuition fees up to date']


### **Decision**

The conservative enrollment-stage feature set contains 22 variables.

`Debtor` and `Tuition fees up to date` are temporarily excluded because their timing still needs to be confirmed. They may contain strong dropout signals, but they should only be used if they would realistically be known at the enrollment-stage prediction point.

Semester 1 and Semester 2 academic variables are excluded from this feature set.

## 8. Create Interpretable Semester 1 Features

Created:
- approval rate = approved / enrolled
- evaluations per enrolled unit
- completion gap = enrolled - approved
- no-approved-units indicator

A separate zero-grade indicator is not created because its zero/nonzero pattern matched zero approved units exactly in EDA.


In [15]:
sem1_enrolled = "Curricular units 1st sem (enrolled)"
sem1_evaluations = "Curricular units 1st sem (evaluations)"
sem1_approved = "Curricular units 1st sem (approved)"

def safe_ratio(numerator, denominator):
    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator, dtype=float),
        where=denominator != 0
    )

df["sem1_approval_rate"] = safe_ratio(
    df[sem1_approved].to_numpy(),
    df[sem1_enrolled].to_numpy()
)

df["sem1_evaluations_per_enrolled"] = safe_ratio(
    df[sem1_evaluations].to_numpy(),
    df[sem1_enrolled].to_numpy()
)

df["sem1_completion_gap"] = df[sem1_enrolled] - df[sem1_approved]

df["sem1_no_approved_units"] = (df[sem1_approved] == 0).astype(int)

engineered_sem1_features = [
    "sem1_approval_rate",
    "sem1_evaluations_per_enrolled",
    "sem1_completion_gap",
    "sem1_no_approved_units",
    "sem1_no_enrolled_units"
]

df[engineered_sem1_features].describe().T


,count,mean,std,min,25%,50%,75%,max
sem1_approval_rate,4424.0,0.697885,0.365247,0.0,0.5,0.833333,1.0,1.0
sem1_evaluations_per_enrolled,4424.0,1.290340,0.567320,0.0,1.0,1.200000,1.6,3.5
sem1_completion_gap,4424.0,1.563969,1.980227,0.0,0.0,1.000000,2.0,9.0
sem1_no_approved_units,4424.0,0.162297,0.368764,0.0,0.0,0.000000,0.0,1.0
sem1_no_enrolled_units,4424.0,0.040687,0.197587,0.0,0.0,0.000000,0.0,1.0


In [14]:
df["sem1_no_enrolled_units"] = (
    df[sem1_enrolled] == 0
).astype(int)

### **Assessment**

Five Semester 1 features were created from the raw academic variables.

The ratio features remain within plausible ranges, and the binary indicators behave as expected.

A separate `sem1_no_enrolled_units` indicator was added so that a zero approval rate can be distinguished between students who enrolled in units but approved none and students who enrolled in no units at all.

## 9. Validate Engineered Features


In [16]:
engineering_checks = pd.DataFrame({
    "feature": engineered_sem1_features,
    "missing_values": [int(df[col].isna().sum()) for col in engineered_sem1_features],
    "min": [df[col].min() for col in engineered_sem1_features],
    "max": [df[col].max() for col in engineered_sem1_features]
})

engineering_checks


,feature,missing_values,min,max
0,sem1_approval_rate,0,0.0,1.0
1,sem1_evaluations_per_enrolled,0,0.0,3.5
2,sem1_completion_gap,0,0.0,9.0
3,sem1_no_approved_units,0,0.0,1.0
4,sem1_no_enrolled_units,0,0.0,1.0


### **Assessment**

All engineered Semester 1 features passed the basic validation checks.

No missing values were introduced, and the observed ranges are plausible. The approval-rate feature remains between 0 and 1, while the two indicator variables remain binary.

The engineered features can therefore be carried forward for further comparison and modeling.

## 10. Check Engineered Features Against Dropout


In [17]:
engineered_by_target = (
    df.groupby("Dropout_binary")[engineered_sem1_features]
      .agg(["mean", "median"])
)

engineered_by_target


sem1_approval_rate           sem1_evaluations_per_enrolled  \
                             mean    median                          mean   
Dropout_binary                                                              
0                        0.846416  1.000000                      1.298837   
1                        0.383994  0.333333                      1.272382   

                         sem1_completion_gap        sem1_no_approved_units  \
                  median                mean median                   mean   
Dropout_binary                                                               
0               1.166667            0.756910    0.0               0.049284   
1               1.285714            3.269529    3.0               0.401126   

                      sem1_no_enrolled_units         
               median                   mean median  
Dropout_binary                                       
0                 0.0               0.034299    0.0  
1                 0.0               0.054187    0.0

### **Assessment**

The engineered Semester 1 features show different levels of separation between dropout and non-dropout students.

`sem1_approval_rate` shows a large difference between the two groups. Non-dropout students approved a much higher proportion of their enrolled units than dropout students.

`sem1_completion_gap` also shows a clear difference, with dropout students having a larger gap between enrolled and approved units.

The `sem1_no_approved_units` indicator is particularly distinct: about 40% of dropout students approved no Semester 1 units, compared with about 5% of non-dropout students.

In contrast, `sem1_evaluations_per_enrolled` and `sem1_no_enrolled_units` show relatively small differences between the groups.

All engineered features will remain available for baseline modeling, but their contribution will be evaluated formally rather than selected based on these descriptive results alone.

## 11. Review Rare Categories

Categories with fewer than 30 observations are flagged. No grouping is applied yet because grouping rules should be learned from training data only.


In [19]:
RARE_COUNT_THRESHOLD = 30

rare_category_rows = []

for col in categorical_features:
    counts = df[col].value_counts(dropna=False)

    for category, count in counts.items():
        if count < RARE_COUNT_THRESHOLD:
            rare_category_rows.append({
                "feature": col,
                "category": category,
                "count": int(count),
                "percentage": round(count / len(df) * 100, 2)
            })

rare_categories = pd.DataFrame(rare_category_rows)

rare_categories.sort_values(
    ["feature", "count"]
).reset_index(drop=True)


,feature,category,count,percentage
0,Application mode,57,1,0.02
1,Application mode,26,1,0.02
2,Application mode,27,1,0.02
3,Application mode,2,3,0.07
4,Application mode,10,10,0.23
...,...,...,...,...
134,Previous qualification,38,7,0.16
135,Previous qualification,4,8,0.18
136,Previous qualification,9,11,0.25
137,Previous qualification,6,16,0.36


In [21]:
rare_summary = (
    rare_categories
    .groupby("feature")
    .agg(
        rare_category_count=("category", "count"),
        students_in_rare_categories=("count", "sum")
    )
    .reset_index()
)

rare_summary["pct_students_in_rare_categories"] = (
    rare_summary["students_in_rare_categories"] / len(df) * 100
).round(2)

rare_summary.sort_values(
    "rare_category_count",
    ascending=False
)

,feature,rare_category_count,students_in_rare_categories,pct_students_in_rare_categories
2,Father's occupation,34,112,2.53
3,Father's qualification,25,102,2.31
5,Mother's occupation,21,116,2.62
6,Mother's qualification,20,89,2.01
7,Nacionality,19,72,1.63
8,Previous qualification,10,79,1.79
0,Application mode,6,32,0.72
4,Marital status,3,35,0.79
1,Course,1,12,0.27


### **Assessment**

Several coded categorical variables contain many low-frequency categories, particularly parental occupation and qualification variables.

Although some features contain a large number of rare category codes, the students represented by these categories make up only a small percentage of the overall dataset. For example, the rare categories within parental occupation variables account for roughly 2–3% of students.

This suggests that rare-category handling may help reduce sparse encoded features without affecting a large share of observations.

No categories are grouped at this stage. The rare-category threshold and grouping rules will be learned from the training data during modeling to avoid information from the test set influencing preprocessing.

## 12. Why Rare Categories Are Not Grouped Yet

Rare-category grouping is a learned preprocessing decision. It will be fitted on training data during modeling to avoid test-set leakage.


## 13. Define Fairness Audit Variables

`Gender` is the primary categorical fairness audit. `Age at enrollment` remains continuous for prediction and may be grouped later for subgroup evaluation.


In [23]:
fairness_audit_features = [
    "Gender",
    "Age at enrollment",
    "Scholarship holder",
    "Educational special needs",
    "International"
]

fairness_audit_features


['Gender',
 'Age at enrollment',
 'Scholarship holder',
 'Educational special needs',
 'International']

### **Decision**

`Gender` and `Age at enrollment` will be the main variables used for later fairness evaluation.

`Scholarship holder` may support socioeconomic subgroup analysis, while `Educational special needs` and `International` will be interpreted more cautiously because their minority groups are small.


These variables are retained for audit purposes and are not automatically removed from modeling.

## 14. Create Modeling Feature Lists

Two modeling windows are preserved:
- Enrollment-stage model
- First-semester model

Semester 2 features remain excluded from both.


In [24]:
semester1_model_features = (
    enrollment_core_features
    + semester1_raw_features
    + engineered_sem1_features
)

print(f"Enrollment core feature count: {len(enrollment_core_features)}")
print(f"Semester 1 model feature count: {len(semester1_model_features)}")

print("\nSemester 2 variables included in enrollment model:",
      any(col in enrollment_core_features for col in semester2_features))

print("Semester 2 variables included in Semester 1 model:",
      any(col in semester1_model_features for col in semester2_features))


Enrollment core feature count: 22
Semester 1 model feature count: 33

Semester 2 variables included in enrollment model: False
Semester 2 variables included in Semester 1 model: False


### **Decision**

The final feature lists preserve the two intended prediction windows.

The enrollment-stage model uses 22 conservative features available at or near enrollment.

The first-semester model uses those same 22 features plus 6 raw Semester 1 variables and 5 engineered Semester 1 features, for a total of 33 predictors.

No Semester 2 variables are included in either model.
The final feature lists preserve the two intended prediction windows.

The enrollment-stage model uses 22 conservative features available at or near enrollment.

The first-semester model uses those same 22 features plus 6 raw Semester 1 variables and 5 engineered Semester 1 features, for a total of 33 predictors.

No Semester 2 variables are included in either model.

## 15. Check for Duplicate Feature Names and Leakage


In [25]:
checks = {
    "Target excluded from enrollment features":
        "Target" not in enrollment_core_features
        and "Dropout_binary" not in enrollment_core_features,

    "Target excluded from Semester 1 features":
        "Target" not in semester1_model_features
        and "Dropout_binary" not in semester1_model_features,

    "Semester 2 excluded from enrollment features":
        not any(col in enrollment_core_features for col in semester2_features),

    "Semester 2 excluded from Semester 1 features":
        not any(col in semester1_model_features for col in semester2_features),

    "Enrollment feature names unique":
        len(enrollment_core_features) == len(set(enrollment_core_features)),

    "Semester 1 feature names unique":
        len(semester1_model_features) == len(set(semester1_model_features))
}

pd.Series(checks, name="passed")


,passed
Target excluded from enrollment features,True
Target excluded from Semester 1 features,True
Semester 2 excluded from enrollment features,True
Semester 2 excluded from Semester 1 features,True
Enrollment feature names unique,True
Semester 1 feature names unique,True


## 16. Build Processed Modeling Tables

The processed tables remain unencoded. Encoding and scaling will be fitted inside the modeling pipeline.


In [27]:
enrollment_model_data = df[
    enrollment_core_features + ["Dropout_binary"]
].copy()

semester1_model_data = df[
    semester1_model_features + ["Dropout_binary"]
].copy()

print("Enrollment dataset:", enrollment_model_data.shape)
print("Semester 1 dataset:", semester1_model_data.shape)


Enrollment dataset: (4424, 23)
Semester 1 dataset: (4424, 34)


### **Assessment**

The processed modeling tables have the expected dimensions.

The enrollment-stage dataset contains 22 predictors plus the binary target, while the Semester 1 dataset contains 33 predictors plus the target.

The difference reflects the addition of 6 raw Semester 1 variables and 5 engineered Semester 1 features.

## 17. Save Processed Data


In [28]:
possible_processed_dirs = [
    Path("../data/processed"),
    Path("data/processed"),
    Path("/content/student-dropout-capstone/data/processed")
]

processed_dir = next(
    (path for path in possible_processed_dirs if path.parent.exists()),
    possible_processed_dirs[0]
)

processed_dir.mkdir(parents=True, exist_ok=True)

enrollment_path = processed_dir / "enrollment_model_base.csv"
semester1_path = processed_dir / "semester1_model_base.csv"

enrollment_model_data.to_csv(enrollment_path, index=False)
semester1_model_data.to_csv(semester1_path, index=False)

print(f"Saved: {enrollment_path}")
print(f"Saved: {semester1_path}")


Saved: ../data/processed/enrollment_model_base.csv
Saved: ../data/processed/semester1_model_base.csv


### **Output**

Two processed modeling datasets were created:

- `enrollment_model_base.csv` for the enrollment-stage model
- `semester1_model_base.csv` for the first-semester model

The files remain unencoded so that scaling and categorical encoding can be fitted later within the modeling pipeline.

## 18. Create a Feature Manifest


In [29]:
manifest_rows = []

all_manifest_features = sorted(set(
    enrollment_core_features
    + semester1_raw_features
    + engineered_sem1_features
    + timing_pending_features
))

for feature in all_manifest_features:
    if feature in engineered_sem1_features:
        source = "Engineered Semester 1"
    elif feature in semester1_raw_features:
        source = "Raw Semester 1"
    elif feature in timing_pending_features:
        source = "Enrollment — timing pending"
    else:
        source = "Enrollment / context"

    manifest_rows.append({
        "feature": feature,
        "source_stage": source,
        "used_enrollment_core": feature in enrollment_core_features,
        "used_semester1_model": feature in semester1_model_features,
        "fairness_audit_feature": feature in fairness_audit_features
    })

feature_manifest = pd.DataFrame(manifest_rows)

manifest_path = processed_dir / "feature_manifest.csv"
feature_manifest.to_csv(manifest_path, index=False)

feature_manifest.head(20)


,feature,source_stage,used_enrollment_core,used_semester1_model,fairness_audit_feature
0,Admission grade,Enrollment / context,True,True,False
1,Age at enrollment,Enrollment / context,True,True,True
2,Application mode,Enrollment / context,True,True,False
3,Application order,Enrollment / context,True,True,False
4,Course,Enrollment / context,True,True,False
5,Curricular units 1st sem (approved),Raw Semester 1,False,True,False
6,Curricular units 1st sem (credited),Raw Semester 1,False,True,False
7,Curricular units 1st sem (enrolled),Raw Semester 1,False,True,False
8,Curricular units 1st sem (evaluations),Raw Semester 1,False,True,False
9,Curricular units 1st sem (grade),Raw Semester 1,False,True,False


### **Assessment**

The feature manifest correctly documents where each variable comes from and which prediction window can use it.

Enrollment-stage features are available to both models, while raw and engineered Semester 1 features are restricted to the first-semester model.

Timing-sensitive variables remain excluded, and fairness-audit variables are clearly identified for later evaluation.

## 19. Feature Engineering Decisions

### Features Created

The following Semester 1 features were created:

- approval rate
- evaluations per enrolled unit
- completion gap
- no-approved-units indicator
- no-enrolled-units indicator

### Features Not Created

A separate zero-grade indicator was not created because its zero/nonzero pattern is identical to zero approved units.

### Rare Categories

Rare categories were identified but not permanently grouped.

Some coded variables contain many rare levels, but those levels account for only a small share of students. Any grouping rule will therefore be learned from the training data during modeling rather than applied to the full dataset in advance.

### Scaling and Encoding

No scaling or one-hot encoding is permanently applied in this notebook.

These transformations will be fitted only on the training data through the modeling pipeline to avoid leakage.

### Timing-Sensitive Features

`Debtor` and `Tuition fees up to date` remain outside the conservative enrollment-stage feature set until their timing can be justified.

### Fairness Variables

`Gender` and `Age at enrollment` will be the main variables used for later fairness evaluation.

`Scholarship holder`, `Educational special needs`, and `International` will be retained for supporting subgroup analysis where sample sizes allow.

### Leakage Controls

Semester 1 variables are excluded from the enrollment-stage model.

Semester 2 variables are excluded from both primary early-warning models.

The processed datasets remain unencoded so that preprocessing can be fitted safely within the modeling workflow.

## 20. Outputs for the Modeling Stage

This notebook produces:

- `data/processed/enrollment_model_base.csv`
- `data/processed/semester1_model_base.csv`
- `data/processed/feature_manifest.csv`

These files remain unencoded so preprocessing can be fitted safely inside train/test and cross-validation pipelines.

---

## Next Notebook

`04_predictive_modeling.ipynb`
